In [39]:
import pandas as pd
import numpy as np

# Actividad 2 Limpieza y Análisis de Datos con Pandas y NumPy
## Parte 1 - Exploración de datos (Pandas)

In [ ]:
df = pd.read_csv('ventas_sucias_5000.csv')
df.head()

,cliente,producto,precio,cantidad,pais,metodo_pago,fecha
0,Maria,Monitor,1326.0,NaN,peru,Efectivo,2024-01-01 00:00:00
1,Luisa,Laptop,55.0,2,chile,Tarjeta,2024-01-01 01:00:00
2,Carlos,Monitor,1203.0,9,Colombia,Efectivo,2024-01-01 02:00:00
3,Luisa,Monitor,1304.0,3,Perú,TRANSFERENCIA,2024-01-01 03:00:00
4,Luisa,Monitor,426.0,6,chile,Tarjeta,2024-01-01 04:00:00


In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cliente      5000 non-null   str    
 1   producto     5000 non-null   str    
 2   precio       4950 non-null   float64
 3   cantidad     4950 non-null   str    
 4   pais         5000 non-null   str    
 5   metodo_pago  5000 non-null   str    
 6   fecha        5000 non-null   str    
dtypes: float64(1), str(6)
memory usage: 273.6 KB


In [20]:
df.describe()

,precio
count,4950.000000
mean,5053.823434
std,63379.982009
min,10.000000
25%,531.000000
50%,1027.000000
75%,1519.000000
max,999999.000000


## Parte 2 - Limpieza de datos (Pandas)

In [ ]:
df.isnull().sum()

cliente         0
producto        0
precio         50
cantidad       50
pais            0
metodo_pago     0
fecha           0
dtype: int64

No podemos asignar un valor como 0 al registro ya que sería perjudicial para el análisis, por lo que es mejor borrarlos ya que son pocos sin embargo, la columna cantidad tiene valores inválidos por lo que los filtramos con to_numeric y luego eliminamos los nulos.


In [ ]:
df['cantidad'] = pd.to_numeric(df['cantidad'], errors='coerce')
df.dropna(subset=['precio', 'cantidad'], inplace=True)
df.info()

<class 'pandas.DataFrame'>
Index: 4853 entries, 1 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cliente      4853 non-null   str    
 1   producto     4853 non-null   str    
 2   precio       4853 non-null   float64
 3   cantidad     4853 non-null   float64
 4   pais         4853 non-null   str    
 5   metodo_pago  4853 non-null   str    
 6   fecha        4853 non-null   str    
dtypes: float64(2), str(5)
memory usage: 303.3 KB


De 5000 registros quedaron 4853 después de la limpieza, por lo que es válido decir que no se afectó en gran manera el volumen del mismo, ahora se procede con el ajuste de los tipos de datos.

In [ ]:
df['cantidad'] = df['cantidad'].astype(int)
df['fecha'] = pd.to_datetime(df['fecha'])
df.info()

<class 'pandas.DataFrame'>
Index: 4853 entries, 1 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   cliente      4853 non-null   str           
 1   producto     4853 non-null   str           
 2   precio       4853 non-null   float64       
 3   cantidad     4853 non-null   int64         
 4   pais         4853 non-null   str           
 5   metodo_pago  4853 non-null   str           
 6   fecha        4853 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(1), str(4)
memory usage: 303.3 KB


### Estandarización

In [31]:
print(df['pais'].unique())
print(df['metodo_pago'].unique())
print(df['producto'].unique())
print(df['cliente'].unique())

<StringArray>
['chile', 'Colombia', 'Perú', 'COL', 'Chile', 'col', 'peru']
Length: 7, dtype: str
<StringArray>
['Tarjeta', 'Efectivo', 'TRANSFERENCIA', 'transferencia']
Length: 4, dtype: str
<StringArray>
['Laptop', 'Monitor', 'Celular', 'Mouse', 'Teclado']
Length: 5, dtype: str
<StringArray>
['Luisa', 'Carlos', 'Juan', 'Maria', 'Pedro', 'Ana']
Length: 6, dtype: str


De las columnas de texto, destacan dos que se pueden tomar como categóricas que tienen valores sucios, al ser pocos valores a limpiar son fáciles de solucionar mapeando con un diccionario

In [33]:
df['pais'] = df['pais'].str.strip().str.lower()
df['metodo_pago'] = df['metodo_pago'].str.strip().str.lower()

print(df['pais'].unique())
print(df['metodo_pago'].unique())

<StringArray>
['chile', 'colombia', 'perú', 'col', 'peru']
Length: 5, dtype: str
<StringArray>
['tarjeta', 'efectivo', 'transferencia']
Length: 3, dtype: str


In [ ]:
mapeo_paises = {
    'col': 'colombia',
    'perú': 'peru'
} # Prefiero quitar la tilde para evitar problemas de codificación
df['pais'] = df['pais'].replace(mapeo_paises)
print(df['pais'].unique())

<StringArray>
['chile', 'colombia', 'peru']
Length: 3, dtype: str


## Parte 3 - Análisis con Pandas

In [35]:
# Columna total
df['total'] = df['precio'] * df['cantidad']
df.head()

,cliente,producto,precio,cantidad,pais,metodo_pago,fecha,total
1,Luisa,Laptop,55.0,2,chile,tarjeta,2024-01-01 01:00:00,110.0
2,Carlos,Monitor,1203.0,9,colombia,efectivo,2024-01-01 02:00:00,10827.0
3,Luisa,Monitor,1304.0,3,peru,transferencia,2024-01-01 03:00:00,3912.0
4,Luisa,Monitor,426.0,6,chile,tarjeta,2024-01-01 04:00:00,2556.0
5,Juan,Laptop,857.0,6,colombia,transferencia,2024-01-01 05:00:00,5142.0


In [36]:
# Analisis de ventas
total_ventas = df['total'].sum()
promedio_venta = df['total'].mean()
venta_maxima = df['total'].max()
venta_minima = df['total'].min()

print(f"Total de ventas: {total_ventas}")
print(f"Promedio de venta: {promedio_venta}")
print(f"Venta máxima: {venta_maxima}")
print(f"Venta mínima: {venta_minima}")

Total de ventas: 101327946.0
Promedio de venta: 20879.444879456005
Venta máxima: 8999991.0
Venta mínima: 10.0


## Parte 4 - NumPy

In [41]:
data = df[['precio', 'cantidad']].to_numpy()
precios = data[:, 0]
cantidades = data[:, 1]
totales = precios * cantidades

## Parte 5 - Análisis con NumPy

In [40]:
# Suma total de ventas con numpy
total_ventas_numpy = np.sum(totales)
print(f"Total de ventas calculado con numpy: {total_ventas_numpy}")

Total de ventas calculado con numpy: 101327946.0


In [42]:
# Promedio de venta con numpy
promedio_venta_numpy = np.mean(totales)
print(f"Promedio de venta calculado con numpy: {promedio_venta_numpy}")

Promedio de venta calculado con numpy: 20879.444879456005


In [43]:
# Venta máxima con numpy
venta_maxima_numpy = np.max(totales)
print(f"Venta máxima calculada con numpy: {venta_maxima_numpy}")

Venta máxima calculada con numpy: 8999991.0


In [45]:
# Ventas superiores a 1000
ventas_superiores_1000 = len(totales[totales > 1000])
print(f"Ventas superiores a 1000: {ventas_superiores_1000}")

Ventas superiores a 1000: 4117


## Parte 6 - Interpretación de resultados

> **¿Los resultados obtenidos tienen sentido?**

> No realmente, son pocos registros y por ejemplo la venta máxima (asumiendo que es en dólares) es de casi 9 millones de dólares y el total de ventas son 101 millones de dólares.

>**¿Detectaste valores sospechosos?**

> Sí, de hecho, como menciono en la pregunta anterior, hay ventas totales de casi 9 millones de dólares, quién sabe cuántas más hay, eso eleva de sobremanera el total.

>**¿El promedio representa correctamente los datos?**

> Huh, pues si, matemáticamente es acertado.

>**¿Qué decisiones tomarías si esta fuera información real de negocio?**

> Pues buscar dónde esconderme porque eso lo van a auditar 100% jajaja, pero hablando en serio, revisaría si estas ventas exageradas son una inconsistencia o si algo extremo realmente pasó ahí.